# De la Regresión Lineal al Deep Learning
### Estadística II — Universidad Nacional de Colombia, Sede Medellín

**Objetivo:** Mostrar que la regresión lineal y logística que ya conocen de R son los bloques fundamentales de todo el machine learning moderno. No estamos cambiando de tema — estamos **generalizando** lo que ya saben.

> *"Una red neuronal profunda con millones de parámetros, en su última capa, tiene... una regresión lineal (o logística). Todo lo que viene antes es aprender las features (caracteristicas o predictoras)."*

---

**Contenido:**
1. Scikit-learn en 5 minutos: el flujo `fit → predict → score`
2. Overfitting en vivo: por qué la inferencia importa
3. Regularización: Ridge, Lasso y selección de variables
4. Gradient Descent: cómo "aprende" un modelo
5. De regresión logística a clasificación en ML
6. NLP con regresión: clasificar texto con herramientas lineales
7. Redes neuronales = regresiones apiladas (PyTorch en 15 líneas)
8. La regresión lineal es imbatible en interpretabilidad


## 0. Setup e importaciones

Ejecuta esta celda primero. Si algo falla, instala con:
```
pip install scikit-learn torch matplotlib seaborn
```


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Estilo de gráficos
plt.rcParams.update({
    'figure.figsize': (10, 5),
    'font.size': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Semilla para reproducibilidad
np.random.seed(42)
print("✅ Todo listo.")


✅ Todo listo.


## 1. Scikit-learn en 5 minutos: `fit → predict → score`

En R ustedes hacen:
```r
modelo <- lm(y ~ x1 + x2, data = datos)
predict(modelo, newdata = nuevos)
summary(modelo)$r.squared
```

En scikit-learn el flujo es **idéntico** para *cualquier* modelo:
```python
modelo = AlgunModelo()
modelo.fit(X_train, y_train)
modelo.predict(X_test)
modelo.score(X_test, y_test)
```

Cambia `AlgunModelo()` y el resto del código no se toca. Veámoslo:


In [5]:
from sklearn.datasets import fetch_california_housing

# Dataset de precios de casas en California (reemplazo moderno del Boston dataset)
housing = fetch_california_housing(as_frame=True)
X = housing.data
y = housing.target  # precio mediano en cientos de miles de USD

print(f"Dimensiones: {X.shape[0]} casas, {X.shape[1]} variables")
print(f"\nVariables: {list(X.columns)}")
print(f"\nPrimeras filas:")
X.head()


Dimensiones: 20640 casas, 8 variables

Variables: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']

Primeras filas:


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [6]:
# Train/test split — concepto que en Estadística II no ven mucho,
# pero es FUNDAMENTAL en ML
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Estandarizar (importante para regularización)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Entrenamiento: {X_train.shape[0]} observaciones")
print(f"Test: {X_test.shape[0]} observaciones")


Entrenamiento: 16512 observaciones
Test: 4128 observaciones


In [7]:
# === LA MAGIA: mismo código, 4 modelos diferentes ===

modelos = {
    "Regresión Lineal (OLS)": LinearRegression(),
    "Ridge (α=1)": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
}

resultados = {}
for nombre, modelo in modelos.items():
    modelo.fit(X_train_s, y_train)          # fit
    y_pred = modelo.predict(X_test_s)        # predict
    r2 = modelo.score(X_test_s, y_test)      # score
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    resultados[nombre] = {"R²": round(r2, 4), "RMSE": round(rmse, 4)}

pd.DataFrame(resultados).T.style.background_gradient(cmap="RdYlGn", subset=["R²"])


,R²,RMSE
Regresión Lineal (OLS),0.575800,0.745600
Ridge (α=1),0.575800,0.745600
Random Forest,0.805000,0.505500
Gradient Boosting,0.775600,0.542200


### 🤔 Pregunta para discutir
¿Por qué Random Forest y Gradient Boosting superan a OLS? ¿Significa que la regresión lineal es inútil?

**Spoiler:** No. Capturan no-linealidades e interacciones automáticamente, pero pierden interpretabilidad. En muchos contextos reales (medicina, banca, legal) **necesitas** los coeficientes y p-valores que da OLS.


## 2. Overfitting en vivo: el peligro de memorizar

Este es quizás el concepto más importante de ML. Vamos a ver cómo un modelo puede tener $R^2 = 1.0$ en entrenamiento y ser **basura** en predicción.


In [8]:
# Datos: relación cuadrática real con ruido
np.random.seed(42)
n = 30
x = np.linspace(0, 4, n)
y_true = 2 + 1.5*x - 0.8*x**2
y = y_true + np.random.normal(0, 0.8, n)

x_plot = np.linspace(-0.2, 4.2, 300)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
grados = [1, 4, 20]
titulos = ["Subajuste (grado 1)", "Buen ajuste (grado 4)", "Sobreajuste (grado 20)"]
colores = ["#e74c3c", "#27ae60", "#8e44ad"]

for ax, grado, titulo, color in zip(axes, grados, titulos, colores):
    # Ajustar polinomio
    poly = PolynomialFeatures(grado)
    X_poly = poly.fit_transform(x.reshape(-1, 1))
    X_plot_poly = poly.transform(x_plot.reshape(-1, 1))

    reg = LinearRegression().fit(X_poly, y)
    y_pred_train = reg.predict(X_poly)
    y_pred_plot = reg.predict(X_plot_poly)

    r2_train = r2_score(y, y_pred_train)

    # Validación cruzada para estimar R² real
    cv_scores = cross_val_score(
        LinearRegression(), X_poly, y, cv=5, scoring='r2'
    )

    ax.scatter(x, y, color='#2c3e50', zorder=5, s=50, label='Datos')
    ax.plot(x_plot, y_pred_plot, color=color, linewidth=2.5, label=f'Polinomio grado {grado}')
    ax.set_ylim(y.min() - 2, y.max() + 2)
    ax.set_title(f"{titulo}", fontsize=13, fontweight='bold')
    ax.text(0.05, 0.95, f"R² train: {r2_train:.3f}
R² CV:    {cv_scores.mean():.3f}",
            transform=ax.transAxes, va='top', fontsize=11,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    ax.legend(loc='lower left', fontsize=9)

plt.suptitle("¿Más complejidad = mejor modelo?  No necesariamente.", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


SyntaxError: EOL while scanning string literal (315400062.py, line 36)

### La lección

| | Entrenamiento | Test/CV |
|---|---|---|
| **Subajuste** | $R^2$ bajo | $R^2$ bajo |
| **Buen ajuste** | $R^2$ razonable | $R^2$ similar |
| **Sobreajuste** | $R^2 \approx 1$ | $R^2$ muy bajo o negativo |

El grado 20 tiene coeficientes **enormes** que oscilan violentamente. Esto motiva directamente la **regularización**.


## 3. Regularización: Ridge, Lasso y selección automática de variables

En OLS minimizamos $\sum(y_i - \hat{y}_i)^2$. ¿Qué pasa si **penalizamos** coeficientes grandes?

| Método | Penalización | Efecto |
|--------|-------------|--------|
| Ridge | $\lambda \sum \beta_j^2$ | Encoge coeficientes, nunca los elimina |
| Lasso | $\lambda \sum |\beta_j|$ | Encoge **y puede eliminar** coeficientes (selección de variables) |
| Elastic Net | Combinación de ambos | Lo mejor de ambos mundos |

La pregunta clave: **¿cuáles coeficientes sobreviven?**


In [ ]:
# Regularización path: cómo se comportan los coeficientes al aumentar alpha
from sklearn.linear_model import lasso_path

# Usamos datos de California Housing
alphas_lasso, coefs_lasso, _ = lasso_path(X_train_s, y_train, alphas=np.logspace(-4, 1, 100))

fig, ax = plt.subplots(figsize=(12, 6))
for i, col in enumerate(X.columns):
    ax.plot(np.log10(alphas_lasso), coefs_lasso[i], linewidth=2, label=col)

ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel("log₁₀(α)  →  más regularización", fontsize=12)
ax.set_ylabel("Valor del coeficiente", fontsize=12)
ax.set_title("Lasso path: ¿cuáles variables sobreviven?", fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
plt.tight_layout()
plt.show()

print("A medida que α crece, Lasso va 'apagando' variables una por una.")
print("Las que sobreviven más son las más informativas.")


In [ ]:
# Comparación de coeficientes: OLS vs Ridge vs Lasso
fig, ax = plt.subplots(figsize=(12, 5))
x_pos = np.arange(len(X.columns))
width = 0.25

ols = LinearRegression().fit(X_train_s, y_train)
ridge = Ridge(alpha=1.0).fit(X_train_s, y_train)
lasso = Lasso(alpha=0.05).fit(X_train_s, y_train)

bars1 = ax.bar(x_pos - width, ols.coef_, width, label='OLS', color='#3498db', alpha=0.85)
bars2 = ax.bar(x_pos, ridge.coef_, width, label='Ridge (α=1)', color='#e67e22', alpha=0.85)
bars3 = ax.bar(x_pos + width, lasso.coef_, width, label='Lasso (α=0.05)', color='#2ecc71', alpha=0.85)

ax.set_xticks(x_pos)
ax.set_xticklabels(X.columns, rotation=45, ha='right')
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_ylabel("Coeficiente (estandarizado)")
ax.set_title("OLS vs Ridge vs Lasso: encogimiento y selección de variables", fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

# Variables eliminadas por Lasso
eliminadas = X.columns[lasso.coef_ == 0].tolist()
print(f"Variables eliminadas por Lasso: {eliminadas if eliminadas else 'ninguna con α=0.05'}")


## 4. Gradient Descent: cómo "aprende" un modelo

En Estadística II resuelven OLS con la ecuación normal: $\hat{\beta} = (X^TX)^{-1}X^Ty$

Pero hay otra forma: **descenso de gradiente**. Empezamos con $\beta$ aleatorios y los vamos ajustando en la dirección que reduce el error:

$$\beta_{nuevo} = \beta_{actual} - \eta \cdot \nabla \text{MSE}(\beta)$$

Esto es **exactamente** lo que hace una red neuronal. La única diferencia es que tiene más parámetros.


In [ ]:
# Gradient Descent para regresión lineal simple — paso a paso
np.random.seed(42)
x_gd = np.random.uniform(0, 10, 50)
y_gd = 3 + 2 * x_gd + np.random.normal(0, 2, 50)

# Parámetros iniciales (random)
beta0, beta1 = 0.0, 0.0
learning_rate = 0.005
n = len(x_gd)

# Guardar historia
historia = {"iter": [], "beta0": [], "beta1": [], "mse": []}

for i in range(200):
    # Predicción
    y_pred = beta0 + beta1 * x_gd

    # Error
    mse = np.mean((y_gd - y_pred)**2)

    # Gradientes (las derivadas parciales del MSE)
    grad_beta0 = -2/n * np.sum(y_gd - y_pred)
    grad_beta1 = -2/n * np.sum((y_gd - y_pred) * x_gd)

    # Actualizar
    beta0 -= learning_rate * grad_beta0
    beta1 -= learning_rate * grad_beta1

    if i % 10 == 0 or i < 5:
        historia["iter"].append(i)
        historia["beta0"].append(beta0)
        historia["beta1"].append(beta1)
        historia["mse"].append(mse)

# Comparar con solución analítica
from numpy.linalg import lstsq
X_gd = np.column_stack([np.ones(n), x_gd])
beta_analitico = lstsq(X_gd, y_gd, rcond=None)[0]

print(f"Solución analítica (ecuación normal):  β₀ = {beta_analitico[0]:.4f}, β₁ = {beta_analitico[1]:.4f}")
print(f"Solución por gradient descent (200 it): β₀ = {beta0:.4f}, β₁ = {beta1:.4f}")
print(f"\n→ ¡Llegan al mismo lugar! Gradient descent es más lento pero escala a millones de parámetros.")


In [ ]:
# Visualización: la superficie de error y el camino del gradient descent
from matplotlib.colors import LogNorm

b0_range = np.linspace(-2, 8, 100)
b1_range = np.linspace(-1, 5, 100)
B0, B1 = np.meshgrid(b0_range, b1_range)
MSE_surface = np.array([[np.mean((y_gd - (b0 + b1*x_gd))**2) for b0 in b0_range] for b1 in b1_range])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Izquierda: contorno con camino
ax = axes[0]
cs = ax.contourf(B0, B1, MSE_surface, levels=30, cmap='YlOrRd_r', alpha=0.8)
ax.plot(historia["beta0"], historia["beta1"], 'b.-', markersize=8, linewidth=1.5, label='Gradient Descent')
ax.plot(beta_analitico[0], beta_analitico[1], 'g*', markersize=20, zorder=5, label='Solución OLS')
ax.set_xlabel("β₀ (intercepto)")
ax.set_ylabel("β₁ (pendiente)")
ax.set_title("Camino del Gradient Descent", fontweight='bold')
ax.legend()
plt.colorbar(cs, ax=ax, label='MSE')

# Derecha: MSE por iteración
ax = axes[1]
ax.plot(historia["iter"], historia["mse"], 'b-o', markersize=5)
ax.set_xlabel("Iteración")
ax.set_ylabel("MSE")
ax.set_title("Convergencia del error", fontweight='bold')
ax.set_yscale('log')

plt.tight_layout()
plt.show()


### Conexión con Deep Learning

El gradient descent que acabamos de ver es **exactamente** el mismo algoritmo que entrena GPT-4, Stable Diffusion, etc. Las diferencias son:

1. Más parámetros (millones → billones)
2. Mini-batches en vez de todos los datos
3. Optimizadores más sofisticados (Adam, AdaGrad)
4. Backpropagation para calcular gradientes eficientemente

Pero la idea central es la misma: **calcular el gradiente del error y dar un pasito en la dirección correcta**.


## 5. De regresión logística a clasificación en ML

Ya conocen la regresión logística como un GLM:
$$\log\left(\frac{p}{1-p}\right) = X\beta$$

En ML, esto es un **clasificador**. Y resulta ser sorprendentemente competitivo.


In [ ]:
from sklearn.datasets import load_breast_cancer

# Dataset clásico: clasificar tumores como benignos o malignos
cancer = load_breast_cancer(as_frame=True)
X_c = cancer.data
y_c = cancer.target  # 0 = maligno, 1 = benigno

print(f"Variables: {X_c.shape[1]}")
print(f"Observaciones: {X_c.shape[0]}")
print(f"\nDistribución de clases:")
print(y_c.value_counts().rename({0: 'Maligno', 1: 'Benigno'}))

X_c_train, X_c_test, y_c_train, y_c_test = train_test_split(X_c, y_c, test_size=0.2, random_state=42)
scaler_c = StandardScaler()
X_c_train_s = scaler_c.fit_transform(X_c_train)
X_c_test_s = scaler_c.transform(X_c_test)


In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Comparar: regresión logística vs modelos de ML
modelos_clf = {
    "Regresión Logística": LogisticRegression(max_iter=5000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
}

print("=" * 55)
print(f"{'Modelo':<25} {'Accuracy':>10} {'CV mean':>10}")
print("=" * 55)

for nombre, mod in modelos_clf.items():
    mod.fit(X_c_train_s, y_c_train)
    acc = mod.score(X_c_test_s, y_c_test)
    cv = cross_val_score(mod, X_c_train_s, y_c_train, cv=5).mean()
    print(f"{nombre:<25} {acc:>10.4f} {cv:>10.4f}")

print("=" * 55)
print("\n→ La regresión logística compite de tú a tú con modelos complejos.")
print("  Y además te da coeficientes interpretables y p-valores.")


In [ ]:
# Los coeficientes de la logística: interpretación directa
log_reg = LogisticRegression(max_iter=5000).fit(X_c_train_s, y_c_train)

coefs = pd.Series(log_reg.coef_[0], index=X_c.columns)
top_10 = coefs.abs().nlargest(10).index

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#e74c3c' if v < 0 else '#27ae60' for v in coefs[top_10]]
coefs[top_10].plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel("Coeficiente estandarizado")
ax.set_title("Top 10 predictores de tumor benigno (Reg. Logística)", fontweight='bold')
ax.axvline(0, color='gray', linestyle='--')

# Anotación
ax.text(0.98, 0.02, "Verde → asociado a benigno
Rojo → asociado a maligno",
        transform=ax.transAxes, ha='right', va='bottom', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
plt.tight_layout()
plt.show()


## 6. NLP con regresión: clasificar texto con herramientas lineales

¿Se puede clasificar sentimiento de texto con regresión logística? **Sí**, y funciona sorprendentemente bien.

La idea: convertir texto a números con **TF-IDF** (frecuencia de términos, ponderada por rareza), y aplicar regresión logística. 100% lineal.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Mini dataset de reseñas (simulado para no depender de descarga)
reseñas = [
    "Esta película es increíble, me encantó la actuación",
    "Terrible, la peor película que he visto en mi vida",
    "Una obra maestra, cinematografía espectacular",
    "Aburrida, predecible y muy larga, no la recomiendo",
    "Excelente historia, los personajes son muy profundos",
    "Pésima dirección, guión horrible y actuaciones malas",
    "Me hizo llorar de emoción, hermosa película",
    "Una basura total, perdí mi tiempo y dinero",
    "Fascinante de principio a fin, 10 de 10",
    "Decepcionante, esperaba mucho más de este director",
    "Genial banda sonora y efectos visuales impresionantes",
    "Insoportable, me dormí a la mitad",
    "Una joya del cine latinoamericano, muy recomendada",
    "Malísima, ni siquiera vale la pena discutirla",
    "Brillante actuación del protagonista, merecido Óscar",
    "Desperdicio de talento, la trama no tiene sentido",
    "Conmovedora y profunda, una experiencia única",
    "Horrible, cliché tras cliché sin originalidad",
    "Perfecta mezcla de humor y drama, la amé",
    "Pretenciosa y vacía, puro estilo sin sustancia",
]
sentimiento = [1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0,1,0]  # 1=positivo, 0=negativo

# TF-IDF: convertir texto a vectores numéricos
vectorizer = TfidfVectorizer(max_features=100)
X_text = vectorizer.fit_transform(reseñas)

print(f"Matriz TF-IDF: {X_text.shape[0]} documentos × {X_text.shape[1]} features")
print(f"\nAlgunas features (palabras): {vectorizer.get_feature_names_out()[:15].tolist()}")
print("\n→ Cada reseña es ahora un vector numérico. ¡Podemos aplicar regresión logística!")


In [ ]:
# Clasificar con regresión logística
clf_texto = LogisticRegression(max_iter=1000).fit(X_text, sentimiento)

# Probar con reseñas nuevas
nuevas_reseñas = [
    "Me encantó, una película espectacular y emocionante",
    "Horrible actuación, muy aburrida y larga",
    "Interesante historia pero le faltó algo más"
]
X_nuevas = vectorizer.transform(nuevas_reseñas)
predicciones = clf_texto.predict(X_nuevas)
probabilidades = clf_texto.predict_proba(X_nuevas)

print("Predicciones de sentimiento:")
print("-" * 60)
for texto, pred, prob in zip(nuevas_reseñas, predicciones, probabilidades):
    etiqueta = "😊 Positivo" if pred == 1 else "😞 Negativo"
    print(f"'{texto}'")
    print(f"  → {etiqueta} (P(positivo) = {prob[1]:.2f})")
    print()

# Palabras más influyentes
coefs_texto = pd.Series(clf_texto.coef_[0], index=vectorizer.get_feature_names_out())
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

coefs_texto.nlargest(10).plot(kind='barh', ax=axes[0], color='#27ae60')
axes[0].set_title("Palabras más positivas", fontweight='bold')

coefs_texto.nsmallest(10).plot(kind='barh', ax=axes[1], color='#e74c3c')
axes[1].set_title("Palabras más negativas", fontweight='bold')

plt.suptitle("Interpretabilidad: ¿qué palabras influyen en la clasificación?", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### Reflexión
- Esto es NLP con herramientas 100% lineales que ya conocen.
- Sistemas de detección de spam, análisis de sentimiento en redes sociales, y clasificación de documentos legales usan exactamente esto.
- Modelos como BERT y GPT hacen lo mismo pero con representaciones más sofisticadas (no TF-IDF sino embeddings contextuales).


## 7. Redes neuronales = regresiones apiladas

Ahora viene la parte más reveladora. Una red neuronal es literalmente:

1. **Capa 1:** Regresión lineal → activación no lineal
2. **Capa 2:** Otra regresión lineal → activación
3. **...**
4. **Última capa:** Regresión lineal (o logística)

Vamos a construir una desde cero con PyTorch y verificarlo.

### Sin activación (lineal pura)
Una red con una capa oculta y activación lineal es **exactamente** una regresión lineal. Lo vamos a demostrar numéricamente.


In [ ]:
import torch
import torch.nn as nn

# === Red neuronal con 1 capa oculta, SIN activación ===
class RedLineal(nn.Module):
    def __init__(self, n_input, n_hidden):
        super().__init__()
        self.capa1 = nn.Linear(n_input, n_hidden)   # regresión lineal 1
        self.capa2 = nn.Linear(n_hidden, 1)          # regresión lineal 2

    def forward(self, x):
        x = self.capa1(x)    # sin activación → composición lineal
        x = self.capa2(x)
        return x

# Datos simples
np.random.seed(42)
X_demo = np.random.randn(100, 3).astype(np.float32)
y_demo = (2*X_demo[:, 0] - 1*X_demo[:, 1] + 0.5*X_demo[:, 2] + 0.3).astype(np.float32)

X_t = torch.tensor(X_demo)
y_t = torch.tensor(y_demo).unsqueeze(1)

# Entrenar la red "profunda" (que en realidad es lineal)
red = RedLineal(3, 10)
optimizer = torch.optim.Adam(red.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

for epoch in range(500):
    pred = red(X_t)
    loss = loss_fn(pred, y_t)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# Comparar con OLS
from sklearn.linear_model import LinearRegression
ols_demo = LinearRegression().fit(X_demo, y_demo)

# La red "profunda" equivale a W2 @ W1 → una sola matriz
W1 = red.capa1.weight.detach().numpy()  # (10, 3)
b1 = red.capa1.bias.detach().numpy()    # (10,)
W2 = red.capa2.weight.detach().numpy()  # (1, 10)
b2 = red.capa2.bias.detach().numpy()    # (1,)

# Coeficientes efectivos: W2 @ W1
coefs_red = (W2 @ W1).flatten()
intercepto_red = (W2 @ b1).item() + b2.item()

print("Coeficientes efectivos de la red (W₂ × W₁):")
print(f"  {coefs_red}")
print(f"  intercepto: {intercepto_red:.4f}")
print(f"\nCoeficientes OLS:")
print(f"  {ols_demo.coef_}")
print(f"  intercepto: {ols_demo.intercept_:.4f}")
print(f"\n→ ¡Son prácticamente iguales! Sin activación no lineal,")
print(f"  apilar capas lineales = una sola regresión lineal.")


### Con activación ReLU: "regresión lineal por pedazos"

Ahora agregamos `ReLU(x) = max(0, x)`. Esto convierte cada neurona en una **regresión lineal que se "enciende" o "apaga"** según la región del espacio.

El resultado: la red puede aprender funciones no lineales como composición de piezas lineales.


In [ ]:
# Red con ReLU vs sin activación — aproximando una función no lineal
class RedConReLU(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

class RedSinReLU(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 32),
            nn.Linear(32, 32),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

# Función no lineal verdadera
x_nl = torch.linspace(-3, 3, 200).unsqueeze(1)
y_nl = torch.sin(x_nl) + 0.3 * x_nl**2

# Entrenar ambas redes
def entrenar(red, x, y, epochs=2000):
    opt = torch.optim.Adam(red.parameters(), lr=0.01)
    loss_fn = nn.MSELoss()
    for _ in range(epochs):
        pred = red(x)
        loss = loss_fn(pred, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return red

torch.manual_seed(42)
red_relu = entrenar(RedConReLU(), x_nl, y_nl)
torch.manual_seed(42)
red_lineal = entrenar(RedSinReLU(), x_nl, y_nl)
ols_simple = LinearRegression().fit(x_nl.numpy(), y_nl.numpy())

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(x_nl.numpy(), y_nl.numpy(), 'k-', linewidth=3, label='f(x) = sin(x) + 0.3x² (verdadera)', alpha=0.7)
ax.plot(x_nl.numpy(), red_relu(x_nl).detach().numpy(), 'r-', linewidth=2.5, label='Red con ReLU (2 capas ocultas)')
ax.plot(x_nl.numpy(), red_lineal(x_nl).detach().numpy(), 'b--', linewidth=2, label='Red sin activación (= regresión lineal)')
ax.plot(x_nl.numpy(), ols_simple.predict(x_nl.numpy()), 'g:', linewidth=2, label='OLS directo')

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("ReLU permite a la red aprender no linealidades — sin ReLU, es solo una recta", fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print("→ Sin activación, no importa cuántas capas tengas: solo puedes dibujar una recta.")
print("  ReLU agrega 'quiebres' que permiten aproximar cualquier función.")
print("  Cada neurona ReLU = una regresión lineal que se activa en una región.")


## 8. La regresión lineal es imbatible en interpretabilidad

Un modelo complejo puede ser más preciso, pero en muchos contextos **no puedes usar una black box**:

- **Medicina:** "¿Por qué le diagnosticaron esto al paciente?"
- **Banca:** "¿Por qué rechazaron mi crédito?"
- **Legal:** "¿Qué evidencia soporta esta decisión?"

La regresión lineal con inferencia (lo que ustedes aprenden en Estadística II) es exactamente lo que la industria necesita para **modelos explicables**.

Técnicas modernas como SHAP reconectan modelos complejos con la interpretabilidad lineal.


In [ ]:
# Demostración: importancia de features en Random Forest vs coeficientes de regresión
from sklearn.inspection import permutation_importance

# Modelos ya entrenados con California Housing
rf_model = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train_s, y_train)
ols_model = LinearRegression().fit(X_train_s, y_train)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Izquierda: coeficientes OLS (interpretables directamente)
coefs_ols = pd.Series(ols_model.coef_, index=X.columns).sort_values()
colors_ols = ['#e74c3c' if v < 0 else '#3498db' for v in coefs_ols]
coefs_ols.plot(kind='barh', ax=axes[0], color=colors_ols)
axes[0].set_title("Regresión Lineal: coeficientes
(interpretación directa)", fontweight='bold')
axes[0].set_xlabel("Efecto sobre el precio (estandarizado)")

# Derecha: importancia por permutación del Random Forest
perm_imp = permutation_importance(rf_model, X_test_s, y_test, n_repeats=10, random_state=42)
imp = pd.Series(perm_imp.importances_mean, index=X.columns).sort_values()
imp.plot(kind='barh', ax=axes[1], color='#9b59b6')
axes[1].set_title("Random Forest: importancia por permutación
(no te dice dirección ni magnitud)", fontweight='bold')
axes[1].set_xlabel("Caída en R² al permutar")

plt.tight_layout()
plt.show()

print("→ OLS te dice CUÁNTO y EN QUÉ DIRECCIÓN afecta cada variable.")
print("  Random Forest solo te dice cuáles son importantes, no cómo.")


## Resumen: ¿qué se llevan de esta demo?

| Concepto de Estadística II | Conexión con ML/DL |
|---|---|
| OLS y ecuación normal | Gradient descent resuelve lo mismo, pero escala |
| $R^2$ y diagnósticos | Validación cruzada, bias-variance tradeoff |
| Selección de variables (stepwise) | Lasso y regularización |
| Regresión logística (GLM) | Clasificador lineal, última capa de redes neuronales |
| Coeficientes e intervalos de confianza | Interpretabilidad, SHAP, modelos explicables |
| Supuestos del modelo lineal | Motivación para modelos no lineales |

### El mensaje central

**No están aprendiendo algo obsoleto.** La regresión lineal y logística son el fundamento sobre el que se construye todo lo demás. Si las entienden bien — inferencia, supuestos, diagnósticos — entienden la mitad de lo que hay detrás del deep learning.

La otra mitad es ingeniería: más datos, más capas, GPUs, y trucos de optimización. Pero la estadística de fondo es la misma.

### ¿Por qué Python y no R?

No es mejor ni peor para estadística. Pero si quieren que sus modelos salgan del notebook y lleguen a producción (APIs, apps web, pipelines de datos), el ecosistema vive en Python. Saber ambos lenguajes los hace más versátiles.

---
*Estadística II — UNAL Medellín, 2026-1*
